# A-Polymer Update Strategy Matrix

This notebook runs a concrete experiment matrix for three constant-food A-polymer networks and four SSA/blended update configurations. The output layout is `outputs/<timestamp>/experiment_matrix/`, containing `test_config.csv/xlsx`, `test_result.csv/xlsx`, trajectories, and per-cell cProfile reports with the top 40 cumulative-time entries.

In [1]:
from pathlib import Path
import sys
import pandas as pd

COMPARE_DIR = Path.cwd()
if COMPARE_DIR.name != 'compare':
    COMPARE_DIR = Path(r'C:/Users/33973/Documents/New project/examples/compare')
sys.path.insert(0, str(COMPARE_DIR))

from experiment_matrix import (
    run_experiment_matrix,
    matrix_run_dir,
    write_config_files,
    load_config_dataframe,
)
from a_polymer_update_matrix import (
    DEFAULT_A_POLYMER_NETWORKS,
    create_a_polymer_update_config,
)


## Configuration Info

Edit this block to control the three networks, wall-clock budget, worker count, and blended parameters.

In [3]:
NETWORKS = list(DEFAULT_A_POLYMER_NETWORKS)
WALL_SECONDS = 60.0
MAX_STEPS = 100_000_000
SEED = 123
T_END = 60.0
WORKERS = 12
PROFILE_LIMIT = 40
OUTPUT_ROOT = COMPARE_DIR / 'outputs'
TIMESTAMP = None  # None creates a timestamped output directory

BLENDED_I1 = 100.0
BLENDED_I2 = 150.0
BLENDED_DT_CLE = 0.00033981
BLENDED_DT_MACRO = 0.00033981

config_info = pd.DataFrame(
    [
        ('networks', NETWORKS),
        ('wall_seconds', WALL_SECONDS),
        ('max_steps', MAX_STEPS),
        ('seed', SEED),
        ('t_end', T_END),
        ('workers', WORKERS),
        ('profile_limit', PROFILE_LIMIT),
        ('output_root', str(OUTPUT_ROOT)),
        ('timestamp', TIMESTAMP),
        ('blended_i1', BLENDED_I1),
        ('blended_i2', BLENDED_I2),
        ('blended_dt_cle', BLENDED_DT_CLE),
        ('blended_dt_macro', BLENDED_DT_MACRO),
    ],
    columns=['parameter', 'value'],
)
display(config_info)


,parameter,value
0,networks,"[polymer_a_len5_a5_catalyzes_a_constant_food, ..."
1,wall_seconds,60.0
2,max_steps,100000000
3,seed,123
4,t_end,60.0
5,workers,12
6,profile_limit,40
7,output_root,c:\Users\33973\Documents\New project\examples\...
8,timestamp,None
9,blended_i1,100.0


## Matrix

Rows are method configurations. Columns are network instances. Each cell is a JSON run specification consumed by `run_experiment_matrix`.

In [4]:
config = create_a_polymer_update_config(
    networks=NETWORKS,
    wall_seconds=WALL_SECONDS,
    max_steps=MAX_STEPS,
    seed=SEED,
    t_end=T_END,
    blended_i1=BLENDED_I1,
    blended_i2=BLENDED_I2,
    blended_dt_cle=BLENDED_DT_CLE,
    blended_dt_macro=BLENDED_DT_MACRO,
)

preview_dir = matrix_run_dir(OUTPUT_ROOT, 'a_polymer_config_preview')
preview_paths = write_config_files(config, preview_dir)
display(config)
preview_paths


,polymer_a_len5_a5_catalyzes_a_constant_food,polymer_a_len6_a6_catalyzes_a_constant_food,polymer_a_len8_a8_catalyzes_a_constant_food
method_config_id,,,
ssa,"{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""..."
blended_global_beta_global_propensity,"{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""..."
blended_local_beta_global_propensity,"{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""..."
blended_local_beta_local_propensity,"{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""...","{""enabled"":true,""function"":""run_matrix_cell"",""..."


{'csv': WindowsPath('c:/Users/33973/Documents/New project/examples/compare/outputs/a_polymer_config_preview/experiment_matrix/test_config.csv'),
 'xlsx': WindowsPath('c:/Users/33973/Documents/New project/examples/compare/outputs/a_polymer_config_preview/experiment_matrix/test_config.xlsx')}

## Run And Results

This block executes the matrix. Every enabled cell writes a trajectory plus `outputs/<timestamp>/experiment_matrix/profiles/<config>__<network>__<method>.prof` and the corresponding `_top40.txt` report.

In [5]:
run_info = run_experiment_matrix(
    config,
    output_root=OUTPUT_ROOT,
    timestamp=TIMESTAMP,
    workers=WORKERS,
    profile=True,
    profile_limit=PROFILE_LIMIT,
)

result = run_info['result']
long_result = run_info['long_result']
display(result)

summary_columns = [
    'status',
    'config_id',
    'network',
    'method',
    'simulation_final_time',
    'n_events',
    'wall_runtime_seconds',
    'trajectory_path',
    'profile_report_path',
    'error',
]
display(long_result[[column for column in summary_columns if column in long_result.columns]])
print(f"run_dir: {run_info['run_dir']}")


[matrix] run_dir=c:\Users\33973\Documents\New project\examples\compare\outputs\20260808_114150\experiment_matrix
[matrix] enabled_tasks=12 workers=12 profile=True profile_limit=40
[matrix] status=ok config=blended_global_beta_global_propensity network=polymer_a_len5_a5_catalyzes_a_constant_food method=gillespie_cle_hybrid sim_time=0.020645739251319822 events=69 wall=60.13924389984459 trajectory=c:\Users\33973\Documents\New project\examples\compare\outputs\20260808_114150\experiment_matrix\trajectories\blended_global_beta_global_propensity__polymer_a_len5_a5_catalyzes_a_constant_food__gillespie_cle_hybrid.npz profile=c:\Users\33973\Documents\New project\examples\compare\outputs\20260808_114150\experiment_matrix\profiles\blended_global_beta_global_propensity__polymer_a_len5_a5_catalyzes_a_constant_food__gillespie_cle_hybrid_top40.txt error=
[matrix] status=ok config=blended_local_beta_global_propensity network=polymer_a_len8_a8_catalyzes_a_constant_food method=gillespie_cle_hybrid sim_ti

,polymer_a_len5_a5_catalyzes_a_constant_food,polymer_a_len6_a6_catalyzes_a_constant_food,polymer_a_len8_a8_catalyzes_a_constant_food
method_config_id,,,
ssa,"{""config_id"":""ssa"",""error"":"""",""final_total_abu...","{""config_id"":""ssa"",""error"":"""",""final_total_abu...","{""config_id"":""ssa"",""error"":"""",""final_total_abu..."
blended_global_beta_global_propensity,"{""config_id"":""blended_global_beta_global_prope...","{""config_id"":""blended_global_beta_global_prope...","{""config_id"":""blended_global_beta_global_prope..."
blended_local_beta_global_propensity,"{""config_id"":""blended_local_beta_global_propen...","{""config_id"":""blended_local_beta_global_propen...","{""config_id"":""blended_local_beta_global_propen..."
blended_local_beta_local_propensity,"{""config_id"":""blended_local_beta_local_propens...","{""config_id"":""blended_local_beta_local_propens...","{""config_id"":""blended_local_beta_local_propens..."


,status,config_id,network,method,simulation_final_time,n_events,wall_runtime_seconds,trajectory_path,profile_report_path,error
0,ok,blended_global_beta_global_propensity,polymer_a_len5_a5_catalyzes_a_constant_food,gillespie_cle_hybrid,0.020646,69,60.139244,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
1,ok,blended_local_beta_global_propensity,polymer_a_len8_a8_catalyzes_a_constant_food,gillespie_cle_hybrid,0.101222,429,60.150403,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
2,ok,blended_global_beta_global_propensity,polymer_a_len6_a6_catalyzes_a_constant_food,gillespie_cle_hybrid,0.036458,282,60.131257,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
3,ok,blended_local_beta_global_propensity,polymer_a_len5_a5_catalyzes_a_constant_food,gillespie_cle_hybrid,0.020647,69,60.151162,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
4,ok,blended_local_beta_global_propensity,polymer_a_len6_a6_catalyzes_a_constant_food,gillespie_cle_hybrid,0.036456,282,60.158353,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
5,ok,blended_local_beta_local_propensity,polymer_a_len5_a5_catalyzes_a_constant_food,gillespie_cle_hybrid,0.020768,80,60.171670,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
6,ok,blended_local_beta_local_propensity,polymer_a_len6_a6_catalyzes_a_constant_food,gillespie_cle_hybrid,0.036552,334,60.163943,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
7,ok,blended_global_beta_global_propensity,polymer_a_len8_a8_catalyzes_a_constant_food,gillespie_cle_hybrid,0.101222,429,60.156549,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
8,ok,blended_local_beta_local_propensity,polymer_a_len8_a8_catalyzes_a_constant_food,gillespie_cle_hybrid,0.100401,517,60.178736,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,
9,ok,ssa,polymer_a_len6_a6_catalyzes_a_constant_food,gillespie_ssa,0.027909,14497,60.154530,c:\Users\33973\Documents\New project\examples\...,c:\Users\33973\Documents\New project\examples\...,


run_dir: c:\Users\33973\Documents\New project\examples\compare\outputs\20260808_114150\experiment_matrix


## Load Existing Config

To run an edited config file later, load `test_config.csv` and pass it to `run_experiment_matrix`.

In [ ]:
# edited_config = load_config_dataframe(preview_paths['csv'])
# run_experiment_matrix(
#     edited_config,
#     output_root=OUTPUT_ROOT,
#     workers=4,
#     profile=True,
#     profile_limit=40,
# )
